# F-JEPA : prédiction d'arrêts non planifiés sur la Tablette mouleuse

**Données.** Export Teeptrak brut de la **Tablette mouleuse**, du 01/09/2025 au 01/09/2026 (`input/export-137588`) :
- **Logs de performance** : chaque cycle, micro-arrêt et arrêt, horodaté à la seconde. On y trouve 4 231 arrêts non planifiés, qualifiés par zone et par composant.
- **Logs d'événements** : une impulsion capteur par cycle, environ toutes les 29 s.

**Question.** À un instant où la machine tourne, peut-on prédire qu'un **arrêt non planifié va commencer dans les 30 prochaines minutes**, à partir des 4 dernières heures ?

**Démarche**
1. Agréger les logs en une série de **pas de 5 minutes** (cadence, dispersion des cycles, micro-arrêts, écarts entre impulsions capteur, arrêts, heure, jour).
2. Définir la cible et faire un découpage **dans le temps** : train de septembre 2025 à avril 2026, test de mai à août 2026.
3. Établir des baselines : règles simples, régression logistique, gradient boosting sur des features manuelles.
4. **Pré-entraîner un JEPA** sur des fenêtres de 4 h (48 tokens de 5 min), sans étiquettes.
5. Évaluer de trois façons : sonde linéaire, score de surprise, fine-tuning supervisé comparé à un Transformer initialisé au hasard. Le tout est répété sur 5 seeds.
6. Faire les statistiques (intervalles de confiance, effet de la durée d'entraînement) et analyser par zone.
7. Simuler le modèle en conditions réelles.
8. Conclure.

Les fichiers Excel d'origine sont seulement lus. Un cache (`cache/*.pkl`) évite de les relire à chaque exécution, car la première lecture prend environ 4 minutes. L'exécution complète (5 seeds, 30 epochs) prend environ 1 h 30 sur CPU. Pour un essai rapide, réduire `SEEDS` et `EPOCHS` (cellule paramètres du JEPA).

**Nouveautés de cette version** : pré-entraînement 2 fois plus long, **5 seeds** avec statistiques (moyenne, écart-type, intervalles de confiance), courbe d'apprentissage, et **simulation de démonstration** en fin de notebook.

## 0. Paramètres

In [ ]:
import os, re, time, copy, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings("ignore", message="enable_nested_tensor")
pd.set_option("display.width", 160)

EXPORT_DIR = Path(os.environ.get("FJEPA_EXPORT", "../input/export-137588"))
CACHE_DIR = Path("cache"); CACHE_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path("results_mouleuse"); RESULTS_DIR.mkdir(exist_ok=True)

BIN = "5min"            # pas de temps d'un token
WINDOW = 48             # 48 x 5 min = 4 h d'historique
HORIZON = 6             # cible : arrêt non planifié dans les 6 prochains pas (30 min)
TRAIN_END = "2026-04-30 23:59"   # train : sept. 2025 -> avril 2026 (8 mois)
TEST_START = "2026-05-01"        # test  : mai -> août 2026 (4 mois)
MIN_STOP_S = 0          # durée minimale d'un arrêt pour compter comme cible (0 = tous)

## 1. Chargement

On renomme les colonnes et on retire les espaces parasites dans `Issue type lv2`. On écarte aussi 2 lignes tronquées en bord d'export, repérables à leur Id entre parenthèses.

In [ ]:
def load_export():
    """Charge les deux types d'exports Teeptrak (avec cache pickle, la lecture Excel prend ~4 min)."""
    pc, ec = CACHE_DIR / "perf.pkl", CACHE_DIR / "events.pkl"
    if pc.exists() and ec.exists():
        return pd.read_pickle(pc), pd.read_pickle(ec)
    perf = pd.concat([pd.read_excel(f, header=1) for f in sorted(EXPORT_DIR.glob("export-*-[0-9]*.xlsx")) if "events" not in f.name])
    ev = pd.concat([pd.read_excel(f) for f in sorted(EXPORT_DIR.glob("export-*-events-*.xlsx"))])
    perf.to_pickle(pc); ev.to_pickle(ec)
    return perf, ev

t0 = time.time()
perf, ev = load_export()
perf.columns = [" ".join(c.replace("\xa0", " ").split()).replace(" ?", "?") for c in perf.columns]
perf = perf.rename(columns={"Commencé à": "debut", "Terminé à": "fin", "Est-ce une perte?": "perte",
                            "Issue type lv2": "lv2", "Niveau 0": "niveau0", "Type de perte": "zone", "Durée": "duree"})
perf["lv2"] = perf["lv2"].fillna("").str.strip()
perf["zone"] = perf["zone"].fillna("").str.strip()
perf = perf[~perf["Id"].astype(str).str.startswith("(")]          # 2 lignes tronquées en bord d'export
perf = perf.sort_values("debut").reset_index(drop=True)
ev = ev.sort_values("Event At").reset_index(drop=True)
print(f"{len(perf):,} lignes de performance, {len(ev):,} impulsions cycle ({time.time()-t0:.0f} s)")
print(f"période {perf.debut.min():%d/%m/%Y} -> {perf.fin.max():%d/%m/%Y}, machine(s) : {perf.Machine.unique()}")
perf.groupby("niveau0", dropna=False)["duree"].agg(nb="count", heures=lambda s: round(s.sum() / 3600, 1))

### Les arrêts non planifiés

Ce sont nos étiquettes de panne : chaque arrêt est qualifié par une zone (`Type de perte`) et un composant (`Issue type lv2`).

In [ ]:
stops = perf[perf.niveau0 == "Arrêts non planifiés"].copy()
print(f"{len(stops)} arrêts non planifiés ; durée médiane {stops.duree.median():.0f} s, "
      f"{(stops.duree >= 300).sum()} >= 5 min, {(stops.duree >= 900).sum()} >= 15 min, {stops.Commentaires.notna().sum()} commentés")
top = (stops.groupby(["zone", "lv2"]).duree.agg(nb="count", heures=lambda s: s.sum() / 3600)
       .sort_values("heures", ascending=False).round(1).head(15))
top

## 2. Série temporelle par pas de 5 minutes

Pour chaque pas de 5 minutes, on calcule :
- **la cadence** : nombre de cycles normaux, durée moyenne, maximale et écart-type des cycles, temps de cycle cible du produit (27 s ou 18 s) ;
- **la dégradation** : nombre et durée des micro-arrêts (« sous-cadence »), nombre de « sur-cadences » ;
- **le capteur** : nombre d'impulsions, écart moyen et maximal entre deux impulsions ;
- **l'état** : secondes en arrêt non planifié et en fermeture, en production ou non ;
- **le calendrier** : heure et jour de la semaine (sin/cos).

In [ ]:
def spread_seconds(df, idx):
    """Secondes passées dans des intervalles [debut, fin) pour chaque pas de temps (gère les intervalles à cheval)."""
    out = np.zeros(len(idx))
    step = pd.Timedelta(BIN)
    t0 = idx[0]
    for s, e in zip(df.debut.values, df.fin.values):
        if e <= s:
            continue
        i0 = int((s - t0) // step); i1 = int((e - t0) // step)
        for i in range(max(i0, 0), min(i1, len(idx) - 1) + 1):
            a = max(s, idx[i]); b = min(e, idx[i] + step)
            out[i] += max((b - a) / np.timedelta64(1, "s"), 0)
    return out

idx = pd.date_range(perf.debut.min().floor(BIN), perf.fin.max().ceil(BIN), freq=BIN)
b = pd.DataFrame(index=idx)
perf["t"] = perf.debut.dt.floor(BIN)
stops["t"] = stops.debut.dt.floor(BIN)
cyc = perf[~perf.perte]
micro = perf[perf.lv2 == "Sous-cadence"]
sur = perf[perf.lv2 == "Sur-cadence"]
b["n_cycles"] = cyc.groupby("t").size()
b["cycle_moy"] = cyc.groupby("t").duree.mean()
b["cycle_max"] = cyc.groupby("t").duree.max()
b["cycle_std"] = cyc.groupby("t").duree.std()
b["n_micro"] = micro.groupby("t").size()
b["micro_s"] = micro.groupby("t").duree.sum()
b["n_sur"] = sur.groupby("t").size()
b["temps_cycle_cible"] = perf.groupby("t")["Temps de cycle"].median()       # 27 s ou 18 s selon le produit
ev["t"] = ev["Event At"].dt.floor(BIN)
gap = ev["Event At"].diff().dt.total_seconds()
ev["gap"] = gap.where(gap < 600)                                           # écart entre impulsions capteur
b["n_impulsions"] = ev.groupby("t").size()
b["impulsion_ecart_moy"] = ev.groupby("t").gap.mean()
b["impulsion_ecart_max"] = ev.groupby("t").gap.max()
b["arret_s"] = spread_seconds(stops, idx)
b["ferme_s"] = spread_seconds(perf[perf.niveau0 == "Fermeture"], idx)
b["arret_debut"] = stops[stops.duree >= MIN_STOP_S].groupby("t").size()
b = b.fillna({c: 0 for c in b.columns if c.startswith(("n_", "micro", "arret", "ferme"))})
b["temps_cycle_cible"] = b["temps_cycle_cible"].ffill().fillna(27)
for c in ["cycle_moy", "cycle_max", "cycle_std", "impulsion_ecart_moy", "impulsion_ecart_max"]:
    b[c] = b[c].fillna(0)
b["en_production"] = (b.n_cycles > 0).astype(float)
hod = b.index.hour + b.index.minute / 60
b["heure_sin"], b["heure_cos"] = np.sin(2 * np.pi * hod / 24), np.cos(2 * np.pi * hod / 24)
b["jour_sin"], b["jour_cos"] = np.sin(2 * np.pi * b.index.dayofweek / 7), np.cos(2 * np.pi * b.index.dayofweek / 7)
print(f"{len(b):,} pas de 5 min, dont {int(b.en_production.sum()):,} avec production")
b.describe().T[["mean", "std", "max"]].round(2)

## 3. Cible et découpage

- **Cible** : au moins un arrêt non planifié commence dans les 30 minutes qui suivent la fin de la fenêtre.
- **Instants évalués** : seulement ceux où la machine tourne (au moins 3 cycles dans le pas courant, pas d'arrêt ni de fermeture en cours). Prédire un arrêt quand la machine est déjà arrêtée n'aurait pas de sens.
- **Découpage dans le temps** : train jusqu'au 30/04/2026, test à partir du 01/05/2026. La standardisation est calculée sur le train.

Il y a beaucoup de positifs (25 à 35 %), car la machine s'arrête en moyenne toutes les 26 minutes de production. Pour lire les résultats, deux repères :
- **ROC-AUC** : 0,5 correspond au hasard.
- **Précision top 10 %** : parmi les 10 % d'instants jugés les plus risqués, la part réellement suivie d'un arrêt. C'est ce qui compte pour une alerte.

In [ ]:
# Cible : au moins un arrêt non planifié commence dans les HORIZON prochains pas.
fut = b.arret_debut[::-1].rolling(HORIZON, min_periods=1).sum()[::-1].shift(-1).fillna(0)
b["cible"] = (fut > 0).astype(float)
# On ne prédit qu'à des instants où la machine tourne (cycles dans le pas courant, pas d'arrêt en cours).
b["predictible"] = (b.n_cycles >= 3) & (b.arret_s < 60) & (b.ferme_s < 60)
FEATS = [c for c in b.columns if c not in ("cible", "predictible", "arret_debut")]
X_all = b[FEATS].astype(np.float32)
X_all[["micro_s", "arret_s", "ferme_s", "cycle_max", "impulsion_ecart_max"]] = np.log1p(X_all[["micro_s", "arret_s", "ferme_s", "cycle_max", "impulsion_ecart_max"]])
train_rows = X_all.index <= pd.Timestamp(TRAIN_END)
mu, sd = X_all[train_rows].mean(), X_all[train_rows].std().replace(0, 1)
Xn = ((X_all - mu) / sd).values.astype(np.float32)

ends = np.arange(WINDOW - 1, len(b) - HORIZON)          # indice du dernier pas de chaque fenêtre
t_end = b.index[ends]
pred_ok = b.predictible.values[ends]
y_all = b.cible.values[ends]
tr = t_end <= pd.Timestamp(TRAIN_END) - pd.Timedelta(minutes=5 * HORIZON)   # pas de cible qui déborde sur le test
te = t_end >= pd.Timestamp(TEST_START)
trl, tel = tr & pred_ok, te & pred_ok
print(f"{len(FEATS)} variables par pas ; instants prédictibles train {trl.sum():,} / test {tel.sum():,}")
print(f"taux de positifs : train {y_all[trl].mean():.1%}, test {y_all[tel].mean():.1%}")

## 4. Baselines

In [ ]:
rows = []
def score(name, p, sel=None):
    yy = y_all[tel]
    k = max(1, int(0.1 * len(p)))
    top = np.argsort(-p)[:k]
    rows.append({"modele": name, "roc_auc": roc_auc_score(yy, p), "pr_auc": average_precision_score(yy, p),
                 "precision_top10%": yy[top].mean()})
    return rows[-1]

def window_feats(ends_idx):
    """Features manuelles : dernier pas, moyennes 30 min / 2 h / 4 h, temps depuis le dernier arrêt."""
    last = Xn[ends_idx]
    def mean_last(k):
        cs = np.cumsum(np.vstack([np.zeros((1, Xn.shape[1]), np.float32), Xn]), axis=0)
        return (cs[ends_idx + 1] - cs[ends_idx + 1 - k]) / k
    stop_cum = np.concatenate([[0], np.cumsum(b.arret_debut.values)])
    n_stop_4h = stop_cum[ends_idx + 1] - stop_cum[ends_idx + 1 - WINDOW]
    last_stop = pd.Series(np.where(b.arret_debut.values > 0, np.arange(len(b)), np.nan)).ffill().values
    since = np.nan_to_num(ends_idx - last_stop[ends_idx], nan=10_000)
    return np.column_stack([last, mean_last(6), mean_last(24), mean_last(WINDOW), n_stop_4h, np.log1p(since)])

def logreg(C=0.1):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, class_weight="balanced"))

In [ ]:
H = window_feats(ends)
since_last = -H[:, -1]
score("temps depuis le dernier arrêt (règle simple)", since_last[tel])
score("nb d'arrêts dans les 4 h précédentes", H[tel, -2])
p_lr = logreg().fit(H[trl], y_all[trl]).predict_proba(H[tel])[:, 1]
score("régression logistique (features manuelles)", p_lr)
gb = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=0).fit(H[trl], y_all[trl])
p_gb = gb.predict_proba(H[tel])[:, 1]
score("gradient boosting (features manuelles)", p_gb)
pd.DataFrame(rows).round(3)

## 5. JEPA

Chaque **pas de 5 minutes est un token**, et une fenêtre de 4 h en contient 48. L'entraînement se déroule ainsi :
- On masque 2 blocs de 20 minutes à 1 heure. Une fois sur deux, un des blocs est placé en fin de fenêtre, pour que le modèle apprenne aussi à « prévoir ».
- L'encodeur de contexte ne voit que les pas visibles.
- Le prédicteur doit retrouver **les embeddings** des pas masqués, produits par l'encodeur cible. Ce dernier est une moyenne mobile exponentielle (EMA) de l'encodeur de contexte.

La perte se calcule dans l'espace latent, pas sur les valeurs brutes. **Aucune étiquette d'arrêt n'est utilisée** pendant cette phase.

In [ ]:
D_MODEL, N_LAYERS, N_HEADS, PRED_LAYERS = 64, 2, 4, 1
EPOCHS, BATCH_SIZE, LR = 30, 256, 1e-3        # 2x plus long que la première version (15 epochs)
EMA_MOMENTUM = (0.99, 0.999)
MASK_BLOCKS, MASK_BLOCK_LEN = 2, (4, 12)      # 2 blocs de 20 min à 1 h masqués
PRETRAIN_STRIDE = 6                           # une fenêtre toutes les 30 min
SEEDS = [0, 1, 2, 3, 4]                       # 5 entraînements indépendants pour avoir des statistiques
CHECKPOINTS = [5, 10, 20, 30]                 # epochs auxquelles on évalue la sonde (l'entraînement long aide-t-il ?)
# Sur un PC portable, compter ~10-15 min par seed. Pour aller plus vite : SEEDS = [0, 1] et EPOCHS = 15.
torch.set_num_threads(max(1, os.cpu_count() or 1))

def get_windows(end_idx):
    off = np.arange(-WINDOW + 1, 1)
    return torch.from_numpy(Xn[end_idx[:, None] + off])

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_feat, d=D_MODEL, layers=N_LAYERS, heads=N_HEADS):
        super().__init__()
        self.inp = nn.Linear(n_feat, d)
        self.pos = nn.Parameter(torch.randn(1, WINDOW, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, layers)
        self.norm = nn.LayerNorm(d)

    def forward(self, x, hide=None):
        """hide : (B, W) bool, True = pas masqué (invisible pour l'attention)."""
        return self.norm(self.tf(self.inp(x) + self.pos, src_key_padding_mask=hide))


class Predictor(nn.Module):
    def __init__(self, d=D_MODEL, layers=PRED_LAYERS, heads=N_HEADS):
        super().__init__()
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, WINDOW, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, layers)
        self.out = nn.Linear(d, d)

    def forward(self, ctx, hide):
        h = torch.where(hide[..., None], self.mask_token.expand_as(ctx), ctx) + self.pos
        return self.out(self.tf(h))


def sample_mask(batch, gen):
    """Blocs contigus masqués ; une fois sur deux, un bloc colle à la fin (le modèle apprend à « prévoir »)."""
    m = np.zeros((batch, WINDOW), dtype=bool)
    for i in range(batch):
        for k in range(MASK_BLOCKS):
            L = gen.integers(MASK_BLOCK_LEN[0], MASK_BLOCK_LEN[1] + 1)
            start = WINDOW - L if (k == 0 and gen.random() < 0.5) else gen.integers(0, WINDOW - L + 1)
            m[i, start:start + L] = True
    return torch.from_numpy(m)


class JEPA(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        self.context = Encoder(n_feat)
        self.target = copy.deepcopy(self.context)
        for p in self.target.parameters():
            p.requires_grad_(False)
        self.predictor = Predictor()

    def loss(self, x, hide):
        with torch.no_grad():
            tgt = F.layer_norm(self.target(x), (D_MODEL,))
        pred = self.predictor(self.context(x, hide), hide)
        return F.smooth_l1_loss(pred[hide], tgt[hide])

    @torch.no_grad()
    def ema_update(self, m):
        for pc, pt in zip(self.context.parameters(), self.target.parameters()):
            pt.mul_(m).add_(pc.detach(), alpha=1 - m)

    @torch.no_grad()
    def embed(self, x):
        h = self.target(x)
        return torch.cat([h[:, -1], h[:, -6:].mean(1), h.mean(1)], dim=1)   # dernier pas, 30 min, 4 h

    @torch.no_grad()
    def surprise(self, x):
        """Erreur de prédiction des 30 dernières minutes à partir des 3 h 30 précédentes (score non supervisé)."""
        hide = torch.zeros(x.shape[:2], dtype=torch.bool); hide[:, -6:] = True
        tgt = F.layer_norm(self.target(x), (D_MODEL,))
        pred = self.predictor(self.context(x, hide), hide)
        return (pred[:, -6:] - tgt[:, -6:]).pow(2).mean((1, 2))

### Pré-entraînement (sur la période train uniquement, 5 seeds)

In [ ]:
def pretrain(end_idx, seed):
    """Pré-entraînement JEPA. Renvoie le modèle final, l'historique de perte et des copies de l'encodeur cible
    aux epochs CHECKPOINTS."""
    torch.manual_seed(seed); gen = np.random.default_rng(seed)
    model = JEPA(Xn.shape[1])
    params = list(model.context.parameters()) + list(model.predictor.parameters())
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.05)
    steps = EPOCHS * int(np.ceil(len(end_idx) / BATCH_SIZE))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=steps, pct_start=0.1)
    step, hist, ckpts = 0, [], {}
    for ep in range(EPOCHS):
        model.train(); tot = 0.0
        perm = gen.permutation(len(end_idx))
        for i in range(0, len(end_idx), BATCH_SIZE):
            x = get_windows(end_idx[perm[i:i + BATCH_SIZE]])
            loss = model.loss(x, sample_mask(len(x), gen))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
            model.ema_update(EMA_MOMENTUM[0] + (EMA_MOMENTUM[1] - EMA_MOMENTUM[0]) * step / steps); step += 1
            tot += loss.item() * len(x)
        hist.append(tot / len(end_idx))
        if ep + 1 in CHECKPOINTS:
            ckpts[ep + 1] = copy.deepcopy(model.target).eval()
        if (ep + 1) % 10 == 0:
            print(f"  seed {seed} epoch {ep + 1:3d}  perte {hist[-1]:.4f}")
    model.eval()
    return model, hist, ckpts

pre_idx = ends[tr][::PRETRAIN_STRIDE]
print(f"{len(pre_idx):,} fenêtres de pré-entraînement, {EPOCHS} epochs, {len(SEEDS)} seeds")
models, histories, checkpoints = {}, {}, {}
t0 = time.time()
for s in SEEDS:
    models[s], histories[s], checkpoints[s] = pretrain(pre_idx, s)
    torch.save(models[s].state_dict(), RESULTS_DIR / f"jepa_mouleuse_seed{s}.pt")
    print(f"seed {s} terminée ({time.time() - t0:.0f} s cumulées)")
for s, h in histories.items():
    plt.plot(range(1, EPOCHS + 1), h, label=f"seed {s}")
plt.xlabel("epoch"); plt.ylabel("perte JEPA (espace latent)"); plt.legend(); plt.show()

## 6. Évaluation

On évalue le JEPA de trois façons, pour chacune des 5 seeds :
- **Sonde linéaire** : on gèle l'encodeur et on entraîne une régression logistique sur ses embeddings. On la mesure aussi **après 5, 10, 20 et 30 epochs** de pré-entraînement, pour voir si un entraînement plus long aide.
- **Score de surprise** : l'erreur du prédicteur sur les 30 dernières minutes, sans aucune étiquette.
- **Embeddings ajoutés aux features manuelles** dans le gradient boosting.

In [ ]:
@torch.no_grad()
def batched(fn, end_idx, bs=2048):
    return torch.cat([fn(get_windows(end_idx[i:i + bs])) for i in range(0, len(end_idx), bs)]).numpy()

def embed_with(encoder):
    def f(x):
        h = encoder(x)
        return torch.cat([h[:, -1], h[:, -6:].mean(1), h.mean(1)], dim=1)
    return f

# 1) Courbe d'apprentissage : sonde linéaire à chaque checkpoint
curve = []
for s in SEEDS:
    for ep, enc in checkpoints[s].items():
        Ztr, Zte = batched(embed_with(enc), ends[trl]), batched(embed_with(enc), ends[tel])
        p = logreg().fit(Ztr, y_all[trl]).predict_proba(Zte)[:, 1]
        curve.append({"seed": s, "epoch": ep, "roc_auc": roc_auc_score(y_all[tel], p)})
curve = pd.DataFrame(curve)

# 2) Modèle final de chaque seed : sonde, surprise, boosting + embeddings
probe_preds, Z_train = [], {}
for s, model in models.items():
    Ztr, Zte = batched(model.embed, ends[trl]), batched(model.embed, ends[tel])
    Z_train[s] = Ztr
    p = logreg().fit(Ztr, y_all[trl]).predict_proba(Zte)[:, 1]
    probe_preds.append(p)
    score(f"JEPA + sonde linéaire (seed {s})", p)
    score(f"JEPA score de surprise, non supervisé (seed {s})", batched(model.surprise, ends[tel]))
    gbz = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=s)
    gbz.fit(np.hstack([H[trl], Ztr]), y_all[trl])
    score(f"boosting features manuelles + embeddings JEPA (seed {s})", gbz.predict_proba(np.hstack([H[tel], Zte]))[:, 1])
p_probe = np.mean(probe_preds, axis=0)
score("JEPA + sonde linéaire, moyenne des 5 seeds (ensemble)", p_probe)

lc = curve.groupby("epoch").roc_auc.agg(["mean", "std", "min", "max"]).round(3)
print("Sonde linéaire selon la durée du pré-entraînement (5 seeds) :")
lc

### Fine-tuning : le pré-entraînement JEPA aide-t-il ?

On entraîne le même Transformer de façon supervisée, avec deux initialisations : **(a)** l'encodeur JEPA, **(b)** une initialisation aléatoire. On utilise la même seed pour les deux, ce qui permet de les comparer deux à deux. Le score sur le test est relevé à chaque epoch.

In [ ]:
FT_EPOCHS = 4     # 2x plus long que la première version, et sur tous les instants du train (2x plus de données)
class Classifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.enc = encoder
        self.head = nn.Sequential(nn.Linear(3 * D_MODEL, 64), nn.GELU(), nn.Linear(64, 1))
    def forward(self, x):
        h = self.enc(x)
        return self.head(torch.cat([h[:, -1], h[:, -6:].mean(1), h.mean(1)], 1)).squeeze(-1)

def finetune(encoder, seed):
    """Fine-tuning supervisé ; renvoie les probabilités test après chaque epoch."""
    torch.manual_seed(seed); gen = np.random.default_rng(seed)
    clf = Classifier(encoder)
    opt = torch.optim.AdamW([{"params": clf.enc.parameters(), "lr": 2e-4}, {"params": clf.head.parameters(), "lr": 1e-3}], weight_decay=0.05)
    idx_tr, ytr = ends[trl], torch.from_numpy(y_all[trl].astype(np.float32))
    pos_w = (1 - ytr.mean()) / ytr.mean()
    out = []
    for ep in range(FT_EPOCHS):
        clf.train(); perm = gen.permutation(len(idx_tr))
        for i in range(0, len(idx_tr), BATCH_SIZE):
            j = perm[i:i + BATCH_SIZE]
            loss = F.binary_cross_entropy_with_logits(clf(get_windows(idx_tr[j])), ytr[j], pos_weight=pos_w)
            opt.zero_grad(); loss.backward(); opt.step()
        clf.eval()
        out.append(batched(lambda x: torch.sigmoid(clf(x)), ends[tel]))
    return out

t0 = time.time()
ft_curve = []
for s in SEEDS:
    for init, enc in [("JEPA", copy.deepcopy(models[s].target)), ("aléatoire", None)]:
        if enc is None:
            torch.manual_seed(100 + s); enc = Encoder(Xn.shape[1])
        preds = finetune(enc, s)
        for ep, p in enumerate(preds, 1):
            ft_curve.append({"seed": s, "init": init, "epoch": ep, "roc_auc": roc_auc_score(y_all[tel], p)})
        name = "initialisé par le JEPA" if init == "JEPA" else "initialisation aléatoire"
        score(f"Transformer supervisé, {name} (seed {s})", preds[-1])
ft_curve = pd.DataFrame(ft_curve)
print(f"fine-tuning : {time.time() - t0:.0f} s")
ft_curve.pivot_table(index="epoch", columns="init", values="roc_auc", aggfunc=["mean", "std"]).round(3)

## 7. Statistiques

### Synthèse sur les 5 seeds (période de test)

In [ ]:
res = pd.DataFrame(rows)
res.to_csv(RESULTS_DIR / "metrics_mouleuse.csv", index=False)
res["famille"] = res.modele.str.replace(r" \(seed \d\)", "", regex=True)
summary = res.groupby("famille", sort=False).agg(
    n=("roc_auc", "size"), roc_auc=("roc_auc", "mean"), ecart_type=("roc_auc", "std"),
    min=("roc_auc", "min"), max=("roc_auc", "max"), pr_auc=("pr_auc", "mean"),
    precision_top10=("precision_top10%", "mean")).round(3).fillna("")
print(f"Test mai -> août 2026 : {tel.sum():,} instants, {y_all[tel].mean():.1%} suivis d'un arrêt dans les 30 min "
      f"(hasard : ROC-AUC 0,5 ; PR-AUC et précision ≈ {y_all[tel].mean():.3f}). n = nombre de seeds.")
summary

### Intervalles de confiance et comparaison appariée

Les intervalles de confiance à 95 % sont obtenus par **bootstrap par journée** : on rééchantillonne des journées entières, parce que les instants d'une même journée sont corrélés. Si deux intervalles se chevauchent largement, la différence entre les deux modèles n'est pas significative.

In [ ]:
# Intervalle de confiance à 95 % par bootstrap par jour (on rééchantillonne des journées entières,
# car les instants d'une même journée sont corrélés).
days = pd.DatetimeIndex(t_end[tel]).normalize()
uniq = days.unique()
day_id = uniq.get_indexer(days)
rng = np.random.default_rng(0)
def boot_ci(p, n=500):
    yy, vals = y_all[tel], []
    groups = [np.where(day_id == k)[0] for k in range(len(uniq))]
    for _ in range(n):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(uniq), len(uniq))])
        vals.append(roc_auc_score(yy[idx], p[idx]))
    return np.percentile(vals, [2.5, 97.5])

ci = []
for name, p in [("gradient boosting (features manuelles)", p_gb), ("régression logistique (features manuelles)", p_lr),
                ("JEPA + sonde linéaire (ensemble 5 seeds)", p_probe)]:
    lo, hi = boot_ci(p)
    ci.append({"modele": name, "roc_auc": roc_auc_score(y_all[tel], p), "IC95 bas": lo, "IC95 haut": hi})
display(pd.DataFrame(ci).round(3))

# Comparaison appariée : même seed, init JEPA vs init aléatoire
last = ft_curve[ft_curve.epoch == FT_EPOCHS].pivot(index="seed", columns="init", values="roc_auc")
last["écart JEPA - aléatoire"] = last["JEPA"] - last["aléatoire"]
display(last.round(3))
print(f"écart moyen {last['écart JEPA - aléatoire'].mean():+.3f} ± {last['écart JEPA - aléatoire'].std():.3f} ; "
      f"JEPA meilleur sur {(last['écart JEPA - aléatoire'] > 0).sum()}/{len(last)} seeds")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fams = [f for f in summary.index if summary.loc[f, "n"] > 1]
data = [res[res.famille == f].roc_auc.values for f in fams]
axes[0].boxplot(data, orientation="horizontal", tick_labels=[f.replace("Transformer supervisé, ", "Transf. ").replace("boosting features manuelles + embeddings JEPA", "boosting + emb. JEPA") for f in fams])
axes[0].axvline(roc_auc_score(y_all[tel], p_gb), color="C1", ls="--", label="gradient boosting")
axes[0].axvline(0.5, color="0.6", ls=":", label="hasard"); axes[0].legend(fontsize=7); axes[0].set_xlabel("ROC-AUC test (5 seeds)")
for s, g in curve.groupby("seed"):
    axes[1].plot(g.epoch, g.roc_auc, "o-", alpha=0.5, label=f"seed {s}")
axes[1].plot(lc.index, lc["mean"], "k-", lw=2, label="moyenne"); axes[1].set_xlabel("epochs de pré-entraînement JEPA")
axes[1].set_ylabel("ROC-AUC sonde linéaire"); axes[1].legend(fontsize=7); axes[1].set_title("L'entraînement long aide-t-il ?")
for init, g in ft_curve.groupby("init"):
    m = g.groupby("epoch").roc_auc.agg(["mean", "std"])
    axes[2].errorbar(m.index, m["mean"], yerr=m["std"], marker="o", capsize=3, label=f"init {init}")
axes[2].set_xlabel("epochs de fine-tuning"); axes[2].set_ylabel("ROC-AUC test"); axes[2].legend(); axes[2].set_title("Fine-tuning (moyenne ± écart-type)")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "statistiques_seeds.png", dpi=110); plt.show()

### Par zone de panne

In [ ]:
# Quelles zones sont les mieux anticipées ? On compare, pour chaque zone, les instants suivis d'un arrêt de cette zone
# à tous les instants négatifs (score du meilleur modèle JEPA et du boosting).
st = stops[["t", "zone"]].copy()
st["pos"] = b.index.get_indexer(st.t)
zone_of = {}
for z, g in st.groupby("zone"):
    m = np.zeros(len(b), bool)
    for p0 in g.pos.values:
        m[max(p0 - HORIZON, 0):p0] = True
    zone_of[z] = m[ends[tel]]
neg = y_all[tel] == 0
out = []
for z, m in zone_of.items():
    pos = m & ~neg
    if pos.sum() < 30:
        continue
    sel = pos | neg
    out.append({"zone": z, "instants positifs": int(pos.sum()),
                "AUC boosting": roc_auc_score(pos[sel], p_gb[sel]), "AUC sonde JEPA": roc_auc_score(pos[sel], p_probe[sel])})
pd.DataFrame(out).sort_values("instants positifs", ascending=False).round(3)

## 8. Simulation : le modèle en conditions réelles

On rejoue la période de test **comme si le modèle tournait en direct** sur la ligne :
- toutes les 5 minutes, le modèle calcule le risque qu'un arrêt non planifié commence dans les 30 minutes ;
- si ce risque dépasse un **seuil**, une **alerte** est envoyée, au maximum une toutes les 30 minutes. Le seuil est fixé sur la période d'entraînement pour qu'environ 10 % des instants déclenchent une alerte ;
- une alerte est **juste** si un arrêt suit dans les 30 minutes. Un arrêt est **anticipé** si une alerte l'a précédé.

Le modèle de démonstration est l'**ensemble des 5 sondes JEPA**. Le gradient boosting et des alertes tirées **au hasard au même rythme** servent de comparaison.

In [ ]:
# Modèle de démonstration : l'ensemble des 5 sondes JEPA. Le boosting est simulé en parallèle pour comparaison.
# Seuil d'alerte : fixé sur la période d'entraînement pour qu'environ 10 % des instants en production déclenchent
# une alerte. Le test n'est jamais utilisé pour le régler.
ALERT_RATE = 0.10
REFRACTAIRE = HORIZON          # après une alerte, on n'en relance pas pendant 30 min
p_probe_train = np.mean([logreg().fit(Z_train[s], y_all[trl]).predict_proba(Z_train[s])[:, 1] for s in SEEDS], axis=0)
p_gb_train = gb.predict_proba(H[trl])[:, 1]
SEUILS = {"JEPA": np.quantile(p_probe_train, 1 - ALERT_RATE), "boosting": np.quantile(p_gb_train, 1 - ALERT_RATE)}
print("Seuils d'alerte :", {k: round(v, 3) for k, v in SEUILS.items()})

test_t = pd.DatetimeIndex(t_end[tel])
risk = {"JEPA": pd.Series(p_probe, index=test_t), "boosting": pd.Series(p_gb, index=test_t)}
stops_test = stops[stops.debut >= pd.Timestamp(TEST_START)].sort_values("debut")

def simuler(scores, seuil, debut=None, fin=None):
    """Rejoue la période comme en temps réel : toutes les 5 min, le modèle émet une alerte si le risque dépasse le seuil
    (au plus une alerte par tranche de 30 min). Une alerte est « juste » si un arrêt non planifié commence dans les 30 min."""
    s = scores if debut is None else scores[debut:fin]
    st = stops_test if debut is None else stops_test[(stops_test.debut >= debut) & (stops_test.debut < fin)]
    alerts, last = [], None
    for t, p in s.items():
        if p >= seuil and (last is None or t - last >= pd.Timedelta(minutes=5 * REFRACTAIRE)):
            alerts.append((t, p)); last = t
    al = pd.DataFrame(alerts, columns=["t", "risque"])
    # la prédiction à t porte sur ]t+5min ; t+35min] (fin du pas courant + 30 min)
    win_end = al.t + pd.Timedelta(minutes=5 * (HORIZON + 1))
    al["arret_suivant"] = [st.debut[(st.debut >= t + pd.Timedelta(minutes=5)) & (st.debut < e)].min() for t, e in zip(al.t, win_end)]
    al["juste"] = al.arret_suivant.notna()
    al["avance_min"] = (al.arret_suivant - al.t).dt.total_seconds() / 60
    st = st.copy()
    st["anticipe"] = [((al.t <= d - pd.Timedelta(minutes=5)) & (al.t > d - pd.Timedelta(minutes=5 * (HORIZON + 1)))).any() for d in st.debut]
    return al, st

def bilan(al, st, n_jours):
    return {"alertes / jour": len(al) / n_jours, "alertes justes": al.juste.mean(),
            "arrêts anticipés": st.anticipe.mean(), "avance médiane (min)": al.avance_min.median(),
            "arrêts >= 5 min anticipés": st[st.duree >= 300].anticipe.mean()}

### Bilan sur toute la période de test

In [ ]:
n_jours = test_t.normalize().nunique()
tab = {}
for name in ["JEPA", "boosting"]:
    al, st = simuler(risk[name], SEUILS[name]); tab[name] = bilan(al, st, n_jours)
# Référence : alertes tirées au hasard, en même nombre que celles du JEPA (moyenne de 20 tirages)
n_alertes_jepa = len(simuler(risk["JEPA"], SEUILS["JEPA"])[0])
rnd = []
for k in range(20):                        # ~15 min ; réduire à range(5) pour aller plus vite
    r = pd.Series(np.random.default_rng(k).random(len(test_t)), index=test_t)
    lo, hi = 0.0, 1.0
    for _ in range(25):                     # on ajuste le seuil pour obtenir le même nombre d'alertes
        mid = (lo + hi) / 2
        if len(simuler(r, mid)[0]) > n_alertes_jepa: lo = mid
        else: hi = mid
    rnd.append(bilan(*simuler(r, hi), n_jours))
tab["hasard (même nb d'alertes)"] = pd.DataFrame(rnd).mean().to_dict()
print(f"Simulation sur toute la période de test ({n_jours} jours de production, {len(stops_test)} arrêts non planifiés)")
pd.DataFrame(tab).T.round(2)

### Une journée rejouée minute par minute

La journée est choisie automatiquement : c'est celle dont le taux d'arrêts anticipés est le plus proche de la moyenne, donc une journée représentative et non la meilleure. Elle se change avec `DEMO_DAY`.

In [ ]:
# Journée de démonstration : la journée de test dont le taux d'arrêts anticipés est le plus proche de la moyenne
# (pas la meilleure journée). Changer DEMO_DAY pour en voir une autre, ex. DEMO_DAY = "2026-07-15".
al_all, st_all = simuler(risk["JEPA"], SEUILS["JEPA"])
per_day = st_all.groupby(st_all.debut.dt.normalize()).agg(n=("anticipe", "size"), taux=("anticipe", "mean"))
per_day = per_day[per_day.n >= 8]
DEMO_DAY = (per_day.taux - st_all.anticipe.mean()).abs().idxmin()
d0, d1 = pd.Timestamp(DEMO_DAY), pd.Timestamp(DEMO_DAY) + pd.Timedelta(days=1)
al_d, st_d = simuler(risk["JEPA"], SEUILS["JEPA"], d0, d1)
print(f"Journée de démonstration : {d0:%A %d/%m/%Y} — {len(st_d)} arrêts non planifiés, {len(al_d)} alertes\n")

# Journal façon console de supervision
log = [(t, f"⚠  ALERTE  risque {p:.0%}" + (f"  → arrêt {(a - t).total_seconds() / 60:.0f} min plus tard" if pd.notna(a) else "  → pas d'arrêt dans les 30 min (fausse alerte)"))
       for t, p, a in zip(al_d.t, al_d.risque, al_d.arret_suivant)]
log += [(r.debut, f"{'✔' if r.anticipe else '✖'}  ARRÊT    {r.zone} / {r.lv2} ({r.duree / 60:.0f} min)" + ("" if r.anticipe else "  — non anticipé"))
        for r in st_d.itertuples()]
for t, msg in sorted(log):
    print(f"{t:%H:%M}  {msg}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
rk = risk["JEPA"][d0:d1]; rb = risk["boosting"][d0:d1]
full = pd.Series(np.nan, index=pd.date_range(d0, d1, freq=BIN, inclusive="left"))
ax.plot(full.index, rk.reindex(full.index), color="C0", lw=1.3, label="risque JEPA")
ax.plot(full.index, rb.reindex(full.index), color="C1", lw=0.8, alpha=0.6, label="risque boosting")
ax.axhline(SEUILS["JEPA"], color="C0", ls="--", lw=1, label="seuil d'alerte JEPA")
prod = b.loc[d0:d1 - pd.Timedelta(minutes=5), "en_production"]
ax.fill_between(prod.index, 0, 1, where=prod.values == 0, color="0.92", step="post", transform=ax.get_xaxis_transform(), label="machine à l'arrêt / fermée")
for r in st_d.itertuples():
    ax.axvline(r.debut, color="C2" if r.anticipe else "C3", lw=1.5, alpha=0.8)
ax.scatter(al_d.t, al_d.risque, marker="v", s=60, color=["C2" if j else "C3" for j in al_d.juste], zorder=5)
ax.plot([], [], color="C2", label="arrêt anticipé / alerte juste"); ax.plot([], [], color="C3", label="arrêt manqué / fausse alerte")
ax.set_ylabel("probabilité d'arrêt dans 30 min"); ax.set_title(f"Simulation temps réel — {d0:%d/%m/%Y}")
ax.legend(fontsize=7, ncol=3, loc="upper left"); plt.tight_layout(); plt.savefig(RESULTS_DIR / "simulation_journee.png", dpi=110); plt.show()

### Animation (écran de supervision)

In [ ]:
# Animation de la journée (GIF enregistré dans results_mouleuse/), comme un écran de supervision qui se met à jour.
ANIMATION = True
if ANIMATION:
    from matplotlib.animation import FuncAnimation, PillowWriter
    from IPython.display import Image as IPImage
    frames = pd.date_range(d0 + pd.Timedelta(hours=4), d1, freq="15min")
    fig, ax = plt.subplots(figsize=(10, 3.6))
    def draw(now):
        ax.clear()
        r = rk.reindex(full.index)[:now]
        ax.plot(r.index, r.values, color="C0", lw=1.3)
        ax.axhline(SEUILS["JEPA"], color="C0", ls="--", lw=1)
        for x in st_d.itertuples():
            if x.debut <= now:
                ax.axvline(x.debut, color="C2" if x.anticipe else "C3", lw=1.5)
        a = al_d[al_d.t <= now]
        ax.scatter(a.t, a.risque, marker="v", s=60, color=["C2" if (j or t + pd.Timedelta(minutes=35) > now) and j else ("0.4" if t + pd.Timedelta(minutes=35) > now else "C3") for t, j in zip(a.t, a.juste)], zorder=5)
        cur = r.dropna()
        etat = "—" if cur.empty else f"{cur.iloc[-1]:.0%}"
        alerte = (not a.empty) and (now - a.t.iloc[-1] <= pd.Timedelta(minutes=30))
        ax.set_title(f"{now:%d/%m %H:%M}   risque actuel : {etat}   " + ("⚠ ALERTE EN COURS" if alerte else "situation normale"),
                     color="C3" if alerte else "k")
        ax.set_xlim(d0, d1); ax.set_ylim(0, 1); ax.set_ylabel("risque d'arrêt (30 min)")
    anim = FuncAnimation(fig, draw, frames=frames, interval=150)
    anim.save(RESULTS_DIR / "simulation_journee.gif", writer=PillowWriter(fps=6), dpi=70)
    plt.close(fig)
    display(IPImage(filename=str(RESULTS_DIR / "simulation_journee.gif")))

## 9. Conclusions (exécution du 08/10/2026, 5 seeds × 30 epochs)

### Un entraînement plus long améliore-t-il les résultats ? Non.

| Modèle (test mai → août 2026) | ROC-AUC (moyenne ± écart-type, 5 seeds) | IC 95 % (bootstrap par jour) |
|---|---|---|
| Régression logistique (features manuelles) | 0,575 | [0,545 ; 0,600] |
| Gradient boosting (features manuelles) | 0,568 | [0,542 ; 0,596] |
| JEPA + sonde linéaire | 0,563 ± 0,005 | — |
| JEPA + sonde, ensemble des 5 seeds | 0,569 | [0,544 ; 0,595] |
| Transformer supervisé, init. JEPA | 0,564 ± 0,005 | — |
| Transformer supervisé, init. aléatoire | 0,574 ± 0,003 | — |
| JEPA score de surprise | 0,509 ± 0,004 | — |

- **Durée du pré-entraînement.** La sonde passe de 0,569 après 5 epochs à 0,563 après 30. Elle n'augmente pas, elle baisse même légèrement : le JEPA a appris ce qu'il pouvait apprendre de ces données dès les premières epochs.
- **Stabilité.** L'écart-type entre seeds est faible (≈ 0,005), donc les résultats ne dépendent pas d'un tirage chanceux.
- **Pré-entraînement JEPA pour le fine-tuning.** Il fait légèrement **moins bien** qu'une initialisation aléatoire : −0,010 ± 0,007, sur 5 seeds sur 5. L'écart est faible mais systématique.
- **JEPA contre baselines.** Les intervalles de confiance se chevauchent presque entièrement. **Aucune différence significative** : tous les modèles se valent autour de 0,57.

### Ce que montre la simulation

Sur les 4 mois de test, avec environ 7 alertes par jour :

| | alertes justes | arrêts anticipés | arrêts ≥ 5 min anticipés | avance médiane |
|---|---|---|---|---|
| JEPA | 46 % | 33 % | 36 % | 15 min |
| Boosting | 44 % | 31 % | 34 % | 14 min |
| Alertes au hasard (même nombre) | 36 % | 24 % | 21 % | 17 min |

Le modèle fait **nettement mieux que le hasard**. Il anticipe 1,4 fois plus d'arrêts, et 1,7 fois plus pour les arrêts longs. Mais une alerte sur deux reste fausse, et deux arrêts sur trois ne sont pas anticipés. En l'état, ce serait au mieux un **indicateur de vigilance** pour l'équipe, pas un outil de maintenance prédictive fiable.

### Pourquoi ça plafonne, et quoi faire

Les données Teeptrak décrivent des **cycles et des arrêts**, pas l'**état physique** de la machine. Ce ne sont pas la durée d'entraînement ni la taille du modèle qui limitent les résultats, c'est l'information disponible. Les pistes, de la plus prometteuse à la moins prometteuse :
1. **Obtenir des capteurs** (automate, vibration, courant moteur, niveaux du pré-égouttage). C'est sur ce type de signal brut qu'un JEPA a un vrai avantage.
2. **Cibler la zone pré-égouttage** (AUC 0,63–0,64), la seule où une dérive précède l'arrêt.
3. Passer à des tokens **cycle par cycle** plutôt qu'à des agrégats de 5 minutes.
